In [23]:
!pip uninstall -y torchvision -q
!pip install -q transformers datasets evaluate scikit-learn accelerate


In [24]:
import numpy as np
import pandas as pd
import torch
import evaluate
from datasets import Dataset, DatasetDict
from sklearn.metrics import confusion_matrix, classification_report
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

In [25]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [26]:
from google.colab import drive
drive.mount('/content/drive')

train_df = pd.read_csv("/content/drive/MyDrive/nlp_dataset/sent_train.csv")
test_df = pd.read_csv("/content/drive/MyDrive/nlp_dataset/sent_valid.csv")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
print("Train shape:", train_df.shape)
print("Validation shape:", test_df.shape)
print(train_df.head())
print(train_df.columns.tolist())

Train shape: (9543, 2)
Validation shape: (2388, 2)
                                                text  label
0  $BYND - JPMorgan reels in expectations on Beyo...      0
1  $CCL $RCL - Nomura points to bookings weakness...      0
2  $CX - Cemex cut at Credit Suisse, J.P. Morgan ...      0
3  $ESS: BTIG Research cuts to Neutral https://t....      0
4  $FNKO - Funko slides after Piper Jaffray PT cu...      0
['text', 'label']


In [28]:
TEXT_COL = "text"
LABEL_COL = "label"

In [29]:
train_df = train_df.rename(columns={TEXT_COL: "text", LABEL_COL: "label"})
test_df = test_df.rename(columns={TEXT_COL: "text", LABEL_COL: "label"})
train_df = train_df[["text", "label"]].dropna()
test_df = test_df[["text", "label"]].dropna()
train_df["label"] = train_df["label"].astype(int)
test_df["label"] = test_df["label"].astype(int)


In [30]:
print("Class distribution (train):")
print(train_df["label"].value_counts())


Class distribution (train):
label
2    6178
1    1923
0    1442
Name: count, dtype: int64


In [31]:
raw_ds = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
    "validation": Dataset.from_pandas(test_df.reset_index(drop=True)),
})
print(raw_ds)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 9543
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2388
    })
})


In [32]:
label_names = ["Bearish", "Bullish", "Neutral"]  # LABEL_0, LABEL_1, LABEL_2
id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in enumerate(label_names)}


#Choose base model

In [33]:
MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

#text cleaning

In [ ]:
import re

def clean_text(text: str) -> str:
    text = re.sub(r"http\S+|www\.\S+", "", text)      # URLs
    text = re.sub(r"@\w+", "", text)                   # mentions
    text = re.sub(r"\s+", " ", text).strip()            # extra whitespace
    return text

def preprocess(batch):
    cleaned = [clean_text(t) for t in batch["text"]]
    tokenized = tokenizer(
        cleaned,
        truncation=True,
        padding="max_length",
        max_length=128,
    )
    tokenized["labels"] = batch["label"]
    return tokenized

encoded_ds = raw_ds.map(preprocess, batched=True, remove_columns=raw_ds["train"].column_names)
encoded_ds.set_format(type="torch")

train_ds = encoded_ds["train"]
val_ds = encoded_ds["validation"]
print(f"Train size: {len(train_ds)} | Validation size: {len(val_ds)}")

Map:   0%|          | 0/9543 [00:00<?, ? examples/s]

In [ ]:
# Load model with 3-class classification head
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
).to(device)

In [ ]:
# Metrics: accuracy + macro F1 (matches project evaluation criteria)
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    macro_f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "macro_f1": macro_f1}

#Training arguments

In [ ]:
#Training arguments
training_args = TrainingArguments(
    output_dir="./bert-financial-sentiment",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],  # avoids overfitting
)


In [16]:
# Train
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.422336,0.349943,0.875209,0.834685
2,0.272327,0.341427,0.886935,0.851554


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.422336,0.349943,0.875209,0.834685
2,0.272327,0.341427,0.886935,0.851554
3,0.159914,0.455159,0.882328,0.849056
4,0.068672,0.535904,0.879397,0.846223


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=2388, training_loss=0.2593143215930242, metrics={'train_runtime': 1068.5101, 'train_samples_per_second': 35.725, 'train_steps_per_second': 2.235, 'total_flos': 2510891345378304.0, 'train_loss': 0.2593143215930242, 'epoch': 4.0})

In [17]:
#Final evaluation on validation set
eval_results = trainer.evaluate()
print("Final validation metrics:", eval_results)

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.068672,0.342130,4,0.886935,0.851554


Final validation metrics: {'eval_loss': 0.3421298563480377, 'eval_accuracy': 0.8869346733668342, 'eval_macro_f1': 0.8515541676413405}


In [ ]:
#  Detailed error analysis — confusion matrix + per-class report
preds_output = trainer.predict(val_ds)
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = preds_output.label_ids

cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix (rows=true, cols=pred):")
print(pd.DataFrame(cm, index=label_names, columns=label_names))

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=label_names, digits=4))

In [19]:
# Save the fine-tuned model
trainer.save_model("./bert-financial-sentiment-final")
tokenizer.save_pretrained("./bert-financial-sentiment-final")



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./bert-financial-sentiment-final/tokenizer_config.json',
 './bert-financial-sentiment-final/tokenizer.json')

In [ ]:

def predict_sentiment(text: str, model=model, tokenizer=tokenizer):
    model.eval()
    cleaned = clean_text(text)
    inputs = tokenizer(cleaned, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
    pred_id = int(np.argmax(probs))
    return {
        "label": id2label[pred_id],
        "probabilities": {id2label[i]: float(p) for i, p in enumerate(probs)},
    }

# Example
print(predict_sentiment("Company X beats earnings expectations, stock surges 12%"))


In [21]:
#  Compare against your RNN/LSTM/GRU baseline
# Fill in the numbers you already recorded from your baseline notebook.
comparison = pd.DataFrame({
    "Model": ["SimpleRNN", "LSTM", "GRU", "BERT (this run)"],
    "Accuracy": [None, None, None, eval_results["eval_accuracy"]],
    "Macro F1": [None, None, None, eval_results["eval_macro_f1"]],
})
print(comparison)
# comparison.to_csv("model_comparison.csv", index=False)

             Model  Accuracy  Macro F1
0        SimpleRNN       NaN       NaN
1             LSTM       NaN       NaN
2              GRU       NaN       NaN
3  BERT (this run)  0.886935  0.851554


In [22]:
save_path = "/content/drive/MyDrive/nlp_dataset/bert-financial-sentiment-final"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("Model saved to:", save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /content/drive/MyDrive/nlp_dataset/bert-financial-sentiment-final
